In [ ]:
import os
import json
from tavily import TavilyClient
from openai import OpenAI

from dotenv import load_dotenv
load_dotenv()


In [34]:
tavily_client = TavilyClient(api_key=os.getenv("TAVILYAI_API_KEY"))
llm = OpenAI()

In [3]:
from pinecone.grpc import PineconeGRPC as Pinecone


pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY_500"))
index = pc.Index("openai-test")

In [4]:
embed = llm.embeddings.create


In [115]:
class retriever:
        def __init__(self, embed, index):
             self.embed = embed
             self.index = index
        def get_data(self,query):
            k=5
            embedding=self.embed( model="text-embedding-3-large",input=query).data[0].embedding

            vecs = self.index.query(
            vector=embedding,
            top_k=k,
            includeMetadata=True,
            include_values=True
        )["matches"]
            ids=[] 
            for match in vecs:
                ids.append(match.id)
            data = self.index.fetch(ids)
            docs = []
            for key in data["vectors"]:
                docs.append(data["vectors"][key]["metadata"]["text"])
            filtered_docs = self.__evaluate(docs, query)
            n = k-len(filtered_docs)
            print(f"length of n {n}")

            if n==0:
                 return filtered_docs
            else:
                 print(f"{n} queries will be submitted")
                 search_docs = self._web_search(query,n)
                 filtered_docs.extend(search_docs)

                 
                 return filtered_docs


        def __evaluate(self,chunks, query):
             res=  llm.chat.completions.create(
                                model="gpt-4o-mini",
                                messages=[
                                    {
                                    "role": "system",
                                    "content": [
                                        {
                                        "text": "Evaluate the relevance between queries and chunks, grading each chunk as \"yes\" if relevant and \"no\" if irrelevant.\n\n# Steps\n\n1. Analyze the given query and each chunk.\n2. Compare the content of the chunk with the context and intent of the query.\n3. Determine the relevance based on logical consistency, contextual alignment, and key theme extraction.\n4. Conclude the evaluation by grading the chunk as \"yes\" if it is relevant to the query, or \"no\" if it is irrelevant.\n\n# Output Format\n\n- A list indicating each chunk's relevance with \"yes\" or \"no\" based on its alignment with the query.\n\n# Examples\n\n- **Query:** \"How do solar panels work?\"\n  - **Chunk 1:** \"The functioning of solar panels involves the conversion of sunlight into electricity.\" \n    - **Relevance:** \"yes\"\n  - **Chunk 2:** \"Solar panels are made from a variety of materials including monocrystalline silicon.\"\n    - **Relevance:** \"yes\"\n  - **Chunk 3:** \"Global oil prices fluctuated significantly last year.\"\n    - **Relevance:** \"no\"\n\n- **Query:** \"What are the health benefits of green tea?\"\n  - **Chunk 1:** \"Green tea contains antioxidants which are beneficial for health.\"\n    - **Relevance:** \"yes\"\n  - **Chunk 2:** \"The manufacturing process of green tea differs from black tea.\"\n    - **Relevance:** \"no\"",
                                        "type": "text"
                                        }
                                    ]
                                    },
                                    {
                                    "role": "user",
                                    "content": [
                                        {
                                        "text": f"Query: {query} \n Chunks:{chunks} ",
                                        "type": "text"
                                        }
                                    ]
                                    }
                                ],
                                response_format={
                                    "type": "json_schema",
                                    "json_schema": {
                                    "name": "evaluated_chunks",
                                    "schema": {
                                        "type": "object",
                                        "required": [
                                        "chunks"
                                        ],
                                        "properties": {
                                        "chunks": {
                                            "type": "array",
                                            "items": {
                                            "type": "object",
                                            "required": [
                                                "chunk",
                                                "evaluation"
                                            ],
                                            "properties": {
                                                "chunk": {
                                                "type": "string",
                                                "description": "A string representing a chunk of data."
                                                },
                                                "evaluation": {
                                                "enum": [
                                                    "yes",
                                                    "no"
                                                ],
                                                "type": "string",
                                                "description": "Evaluation result which can either be 'yes' or 'no'."
                                                }
                                            },
                                            "additionalProperties": False
                                            },
                                            "description": "A list of evaluated chunks."
                                        }
                                        },
                                        "additionalProperties": False
                                    },
                                    "strict": True
                                    }
                                },
                                temperature=0,
                                max_completion_tokens=2049,
                                top_p=1,
                                frequency_penalty=0,
                                presence_penalty=0
                                )
             filtered_docs =[]
             r = json.loads(res.choices[0].message.content)
             for content in r["chunks"]:
                  if content["evaluation"] == "yes":
                       filtered_docs.append(content["chunk"])
             return filtered_docs
        
        def _web_search(self, query, n=5):
             print(f"started evalueation n{n}")
             response = tavily_client.search(f"For MOHAP (Ministry of Health in the UAE): {query}",max_results=n)["results"]
             return [item["content"] for item in response]


In [116]:
openai_retreiver = retriever(embed, index)

In [118]:
# ## Test
# query="what is the the service i use to register as new practicing doctor in the UAE "
# res = openai_retreiver.get_data(query)
# print(res)
# print(type(res))

length of n 1
1 queries will be submitted
started evalueation n1
['doctor to practice in the medical or dental profession for a limited period of time in a private health facility.Service Process1Login to the MOHAP website or smart app using the UAE PASS to apply for the service.2The customer (facility) must login through the account of the licensed facility and provide all the required information and documents as per the type of license.3The customer must refer the application to the Ministry of Health and Prevention.4The employee concerned will check the application. If the', 'including physicians and dentists of various specialties and levels.Service Process1Login to the MOHAP website or smart app using the UAE PASS to apply for the service.2Fill in the information, attach the required documents, and submit the application.3If the requirements are met, the application will be approved and the customer will pay the service fees.4The license will be issued electronically and sent via